In [1]:
%matplotlib inline 

In [2]:
# Imports
import os
import pathlib
import urllib.request

# Site name for Salvus Flow. Uses env var if available, otherwise falls back to local site name.
SALVUS_FLOW_SITE_NAME = os.environ.get("SALVUS_FLOW_SITE_NAME", "salome_remote_2")
PROJECT_DIR = "simulation_wavefield_experiment" 

# Conservative default to reduce license-seat demand during unstable license-server periods.
RANKS_PER_JOB = 4


def check_license_server_reachable(url="https://l.mondaic.com/licensing_server", timeout=10):
    """Fail fast if licensing endpoint is unreachable to avoid long hanging jobs."""
    try:
        with urllib.request.urlopen(url, timeout=timeout):
            return True
    except Exception as exc:
        raise RuntimeError(
            f"Licensing server not reachable ({url}). Retry later or check network/VPN. Original error: {exc}"
        ) from exc


# Add code to keep .gitignore updated to ignore salvus files
gitignore_path = pathlib.Path("..") / ".gitignore"
with open(gitignore_path, "r+") as f:
    contents = f.read()
    if PROJECT_DIR not in contents:
        f.write(f"\n{PROJECT_DIR}/\n")


import numpy as np
import salvus.namespace as sn
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output

#for plotting of wiggles, traces 
from scipy import signal

# For animation plot
from IPython.display import HTML
from matplotlib.collections import PolyCollection
import threading
import matplotlib
matplotlib.use("Agg")
from scipy.interpolate import griddata

--> Server: 'https://l.mondaic.com/licensing_server', User: 'salome.bachmann', Group: 'ETHZ_ERDW_EEG'.
--> Negotiating 1 license instance(s) for 'SalvusMesh' [license version 1.0.0] for 1 seconds ...
--> Success! [Total duration: 0.19 seconds]


In [3]:
# Setup of the model domain as a box (same as research module)
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=400, y0=0, y1=3)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

Accordion()

In [4]:

# Layered model setup: three layers ordered as snow, slab, air (top to bottom).

x_min, x_max = 0.0, 400.0

# Geometry (y-coordinates: higher = higher depth, measured downward):
# Snow: from y=3.0 m (top) to y=2.25 m
# Slab: from y=2.25 m to y=1.5 m  
# Air:  from y=1.5 m to y=0.0 m (bottom)

slab_top = 3.0
slab_bottom = 2.25
wl_top = 2.25
wl_bottom = 1.5
air_top = 1.5
air_bottom = 0.0

# Boundaries from top to bottom -> 3 layers.
# Must be strictly ordered: top to bottom (decreasing y).
layers_x = [
    np.array([x_min, x_max]),  # snow top boundary
    np.array([x_min, x_max]),  # snow-slab interface
    np.array([x_min, x_max]),  # slab-air interface
    np.array([x_min, x_max]),  # air bottom boundary
]
layers_y = [
    np.array([slab_top, slab_top]),      # y = 3.0
    np.array([slab_bottom, slab_bottom]),  # y = 2.25
    np.array([wl_bottom, wl_bottom]),  # y = 1.5
    np.array([air_bottom, air_bottom]),    # y = 0.0
]

# Material parameters by region index [snow, slab, air].
vp = np.array([332.0, 300.0, 300.0]) 
vs = np.array([150.0, 150.0, 0.0])   
rho = np.array([180.0, 180.0, 1.225])

interpolation_styles = ["linear"] * len(layers_x)
splines = sn.toolbox.get_interpolating_splines(layers_x, layers_y, kind=interpolation_styles)
 
max_frequency = np.percentile([vp[0], vp[1], vp[2]], 95) # set this as the 95th percentile of the expected frequency content
print(f"Max frequency for meshing: {max_frequency:.1f} Hz")
# One per layer pair; last value keeps meshing stable above acoustic air.
slowest_velocities = np.array([150.0, 150.0,  150.0])

mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=x_min,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=2,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    absorbing_boundaries=(["x0", "x1", "y0"], 10.0),
)
mesh = np.sum(mesh)
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)

nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    idx = np.where(mesh.elemental_fields["region"] == _i)
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

mesh_3layer = sn.toolbox.detect_fluid(mesh)
print("Three-layer mesh built.")
print(f"  SLab: y = [{slab_top:.2f}, {slab_bottom:.2f}] m, vs={vs[0]:.0f} m/s")
print(f"  Weak layer: y = [{wl_top:.2f}, {wl_bottom:.2f}] m, vs={vs[1]:.0f} m/s")
print(f"  Air layer:  y = [{air_top:.2f}, {air_bottom:.2f}] m, vs={vs[2]:.0f} m/s")


Max frequency for meshing: 328.8 Hz
Three-layer mesh built.
  SLab: y = [3.00, 2.25] m, vs=150 m/s
  Weak layer: y = [2.25, 1.50] m, vs=150 m/s
  Air layer:  y = [1.50, 0.00] m, vs=0 m/s


In [ ]:
# Clear out any old reference configurations
ref_sim_name = "sim_2d_layered_moving_source_REFERENCE"
ref_event_name = "event_single_reference_source"

if ref_event_name in p.events.list():
    p.events.delete(event_name=ref_event_name)

# Setup of reference source at the starting point (30.0m)

src_ref = sn.simple_config.source.cartesian.MomentTensorPoint2D(
    x=30.0,
    y=2.625,
    mxx=0.0,
    myy=-1.54e+01,
    mxy=0.0,

)

p.add_to_project(sn.Event(event_name=ref_event_name, sources=[src_ref]))

p.add_to_project(
    sn.UnstructuredMeshSimulationConfiguration(
        name=ref_event_name,
        unstructured_mesh=mesh_3layer,
        event_configuration=sn.EventConfiguration(
            wavelet=sn.simple_config.stf.Ricker(center_frequency=20.0),
            waveform_simulation_configuration=sn.WaveformSimulationConfiguration(
                end_time_in_seconds=3.0
            ),
        ),
    ),
    overwrite=True
)


[2026-06-11 14:52:52,446] INFO: A source for event `event_single_reference_source` has a source time function. Events in the project cannot have a source time function (they are frequency dependent). The source time function has thus been stripped.


In [ ]:

print("Launching single master simulation run...")
p.simulations.launch(
    simulation_configuration=ref_event_name,
    events=[ref_event_name],
    site_name=SALVUS_FLOW_SITE_NAME, 
    ranks_per_job=RANKS_PER_JOB,
    extra_output_configuration={
        "volume_data": {
            "sampling_interval_in_time_steps": 50,
            "fields": ["velocity", "displacement"],
        },
    },
)

p.simulations.query(block=True)
print("\nMaster simulation run complete! Ready for rapid local array processing.")

Launching single master simulation run...
[2026-06-11 14:52:52,817] INFO: Submitting job ...
Uploading 1 files...

🚀  Submitted job_2606111452890154_e9e3b3d861@salome_remote_2


VBox()

KeyboardInterrupt: 

In [ ]:
# Array geopetry
spacing = 5.0
x_positions = np.arange(30.0, 270.0 + spacing, spacing) # 30m to 270m
target_vprop = 90.0  # Rupture speed (m/s)
dt_src = spacing / target_vprop

# Load wavefield
sim_dir = pathlib.Path(p.simulations.get_simulation_output_directory(simulation_name="sim_2d_layered_moving_source_REFERENCE"))
vol_file = list(sim_dir.rglob("volume_data_output.h5"))[0]

vel_wo = wavefield_output.WavefieldOutput.from_file(vol_file, "velocity", "volume")

# Generate the base evaluation grid mesh
x_grid = np.linspace(0, 400, 401)
t_grid = np.linspace(0, 3, 101)
y_surface = 1.5

ref_xr = wavefield_output.wavefield_output_to_xarray(vel_wo, points=[x_grid, t_grid])

# Resolve actual dataset dimension name tags dynamically
coords_set = set(ref_xr.coords)
dims_set = set(ref_xr.dims)
x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set))
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set))
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set))
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set))

# Extract the surface velocity data vectors (Horizontal vx & Vertical vy)
ref_vx = ref_xr.isel({c_name: 0}).sel({y_name: y_surface}, method="nearest")
ref_vy = ref_xr.isel({c_name: 1}).sel({y_name: y_surface}, method="nearest")

# Shift-and-Sum Local Superposition Pipeline
print(f"Applying space-time shifts locally for {len(x_positions)} virtual sources...")
combined_vx_data = np.zeros((len(t_grid), len(x_grid)))
combined_vy_data = np.zeros((len(t_grid), len(x_grid)))

base_x_source = x_positions[0] # 30.0m is our zero baseline

for i, x_pos in enumerate(x_positions):
    spatial_shift = x_pos - base_x_source # How many meters to slide down the line
    temporal_shift = 0.3 + (i * dt_src)  # exact sequential time delay profile
    
    # Calculate what coordinates this shifted slice would look like relative to our baseline data
    shifted_x = ref_vx[x_name] - spatial_shift
    shifted_t = ref_vx[t_name] - temporal_shift
    
    # Use fast local interpolation to extract the shifted profile instantly in memory
    vx_shifted = ref_vx.interp({x_name: shifted_x, t_name: shifted_t}, method="linear", kwargs={"fill_value": 0.0})
    vy_shifted = ref_vy.interp({x_name: shifted_x, t_name: shifted_t}, method="linear", kwargs={"fill_value": 0.0})
    
    # Accumulate the matrices
    combined_vx_data += vx_shifted.transpose(t_name, x_name).values
    combined_vy_data += vy_shifted.transpose(t_name, x_name).values

print("Superposition complete with flawless custom time shifts!")

# --- 4. Render the Tilted Wavefront Plots ---
vmax = np.percentile(np.abs([combined_vx_data, combined_vy_data]), 95)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot vx
im_vx = axes[0].pcolormesh(x_grid, t_grid, combined_vx_data, shading="gouraud", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].invert_yaxis()
axes[0].axvspan(x_positions[0], x_positions[-1], color="teal", alpha=0.08, label="Rupture Track")
axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlim(0, 400)
axes[0].set_title("Velocity(vx): Horizontal Component (Shift-and-Sum Mode)")
plt.colorbar(im_vx, ax=axes[0], label="Velocity (vx)")

# Plot vy
im_vy = axes[1].pcolormesh(x_grid, t_grid, combined_vy_data, shading="gouraud", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].invert_yaxis()
axes[1].axvspan(x_positions[0], x_positions[-1], color="teal", alpha=0.08)
axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("Time (s)")
axes[1].set_xlim(0, 400)
axes[1].set_title("Velocity(vy): Vertical Component (Shift-and-Sum Mode)")
plt.colorbar(im_vy, ax=axes[1], label="Velocity (vy)")

plt.tight_layout()
display(fig)
plt.close(fig)